In [22]:
from statemachine import State, StateMachine
import simpy
import functools


def simpy_process(func):
  @functools.wraps(func)
  def wrapper_simpy_process(*args, **kwargs):
    p = args[0].env.process(func(*args, **kwargs))
  return wrapper_simpy_process
 
class TrafficLight(StateMachine):
  red = State(initial=True)
  green = State()
  yellow = State()
  cycle = green.to(yellow) | yellow.to(red) | red.to(green)

  def __init__(self, env, name):
    self.env = env
    self.name = name
    StateMachine.__init__(self)

  @simpy_process
  def on_enter_red(self):
    yield self.env.timeout(30)
    print(f"{self.name} done RED: {self.env.now}")
    self.cycle()
  
  @simpy_process
  def on_enter_green(self):
    yield self.env.timeout(25)
    print(f"{self.name} done GREEN: {self.env.now}")
    self.cycle()
  
  @simpy_process
  def on_enter_yellow(self):
    yield self.env.timeout(5)
    print(f"{self.name} done YELLOW: {self.env.now}")
    self.cycle()

def main():
  env = simpy.Environment()
  t1 = TrafficLight(env, "Water St.")
  t2 = TrafficLight(env, "Main St.")
  env.run(until=200)

if __name__ == '__main__':
  main()

Water St. done RED: 30
Main St. done RED: 30
Water St. done GREEN: 55
Main St. done GREEN: 55
Water St. done YELLOW: 60
Main St. done YELLOW: 60
Water St. done RED: 90
Main St. done RED: 90
Water St. done GREEN: 115
Main St. done GREEN: 115
Water St. done YELLOW: 120
Main St. done YELLOW: 120
Water St. done RED: 150
Main St. done RED: 150
Water St. done GREEN: 175
Main St. done GREEN: 175
Water St. done YELLOW: 180
Main St. done YELLOW: 180
